# Observability: Evaluation Pipeline

----

This notebook demonstrates a **stage-by-stage evaluation pipeline** for a Semantic Kernel-based chat application, using Application Insights logs and the **Azure AI Evaluation SDK (MS Foundry)**.

You will learn how to:

- **Configure Application Insights** telemetry for Semantic Kernel
- **Build an SK orchestration layer**: intent classification → agent routing → function calling → answer generation
- **Parse Application Insights logs** into structured evaluation datasets
- **Run custom evaluators**: intent accuracy (exact match), agent relevance, method relevance
- **Run pre-built evaluators**: similarity, groundedness via Azure AI Evaluation SDK
- **Aggregate results** into a unified evaluation dashboard

## Table of Contents

- [Setup](#setup)
- [Part 1: Application Insights + Semantic Kernel Telemetry](#part-1-application-insights--semantic-kernel-telemetry)
- [Part 2: Semantic Kernel Orchestration Layer](#part-2-semantic-kernel-orchestration-layer)
- [Part 3: Build Evaluation Dataset from App Insights Logs](#part-3-build-evaluation-dataset-from-app-insights-logs)
- [Part 4: Evaluation Step 1 – Intent Classification (Exact Match)](#part-4-evaluation-step-1--intent-classification-exact-match)
- [Part 5: Evaluation Step 2 – Agent Relevance](#part-5-evaluation-step-2--agent-relevance)
- [Part 6: Evaluation Step 3 – Method Relevance](#part-6-evaluation-step-3--method-relevance)
- [Part 7: Evaluation Step 4 – Similarity & Groundedness (Pre-built)](#part-7-evaluation-step-4--similarity--groundedness-pre-built)
- [Part 8: Evaluation Summary Dashboard](#part-8-evaluation-summary-dashboard)
- [Wrap-up](#wrap-up)

## Setup

This notebook reuses the configuration file (`.foundry_config.json`) created by `0_setup/1_setup.ipynb`.

### Prerequisites

| Variable | Description |
|----------|-------------|
| `APPLICATIONINSIGHTS_CONNECTION_STRING` | Azure Application Insights connection string |
| `AZURE_OPENAI_ENDPOINT` | Azure OpenAI endpoint URL |
| `AZURE_OPENAI_API_KEY` | Azure OpenAI API key |
| `AZURE_OPENAI_CHAT_DEPLOYMENT_NAME` | Deployment name (e.g., `gpt-4.1`) |
| `AZURE_AI_PROJECT_ENDPOINT` | Azure AI Foundry project endpoint |

In [34]:
# Environment setup and PATH configuration
import json
import os
import subprocess
import csv
from typing import Dict, List, Any
from dataclasses import dataclass
from dotenv import load_dotenv

load_dotenv(override=True)

# Ensure the notebook kernel can find Azure CLI on PATH
possible_paths = [
    '/opt/homebrew/bin',
    '/usr/local/bin',
    '/usr/bin',
    '/home/linuxbrew/.linuxbrew/bin',
]

az_path = None
try:
    result = subprocess.run(['which', 'az'], capture_output=True, text=True)
    if result.returncode == 0:
        az_path = os.path.dirname(result.stdout.strip())
        print(f'🔍 Azure CLI found: {result.stdout.strip()}')
except Exception:
    pass

paths_to_add: list[str] = []
if az_path and az_path not in os.environ.get('PATH', ''):
    paths_to_add.append(az_path)
else:
    for path in possible_paths:
        if os.path.exists(path) and path not in os.environ.get('PATH', ''):
            paths_to_add.append(path)

if paths_to_add:
    os.environ['PATH'] = ':'.join(paths_to_add) + ':' + os.environ.get('PATH', '')
    print(f"✅ Added to PATH: {', '.join(paths_to_add)}")
else:
    print('✅ PATH looks good already')

🔍 Azure CLI found: /anaconda/envs/azureml_py38/bin//az
✅ PATH looks good already


In [6]:
# Load Foundry project settings
from azure.identity import DefaultAzureCredential

config_file = '../0_setup/.foundry_config.json'
try:
    with open(config_file, 'r', encoding='utf-8') as f:
        config = json.load(f)
except FileNotFoundError as e:
    print(f"⚠️ Could not find '{config_file}'.")
    print('💡 Run 0_setup/1_setup.ipynb first to create it.')
    raise e

FOUNDRY_NAME = config.get('FOUNDRY_NAME')
RESOURCE_GROUP = config.get('RESOURCE_GROUP')
LOCATION = config.get('LOCATION')
PROJECT_NAME = config.get('PROJECT_NAME', 'proj-default')
AZURE_AI_PROJECT_ENDPOINT = config.get('AZURE_AI_PROJECT_ENDPOINT')

AZURE_OPENAI_ENDPOINT = os.environ.get("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.environ.get("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_CHAT_DEPLOYMENT_NAME = os.environ.get(
    "AZURE_OPENAI_CHAT_DEPLOYMENT_NAME", "gpt-4.1"
)
AZURE_OPENAI_API_VERSION = os.environ.get(
    "AZURE_OPENAI_API_VERSION", "2025-03-01-preview"
)
CONNECTION_STRING = os.environ.get("APPLICATIONINSIGHTS_CONNECTION_STRING")

os.environ['FOUNDRY_NAME'] = FOUNDRY_NAME or ''
os.environ['LOCATION'] = LOCATION or ''
os.environ['RESOURCE_GROUP'] = RESOURCE_GROUP or ''
os.environ['AZURE_SUBSCRIPTION_ID'] = config.get('AZURE_SUBSCRIPTION_ID', '')

credential = DefaultAzureCredential()

print(f"✅ Loaded settings from '{config_file}'.")
print(f"📌 Foundry: {FOUNDRY_NAME} | Region: {LOCATION}")
print(f"📌 Project endpoint: {AZURE_AI_PROJECT_ENDPOINT}")
print(f"📌 Chat deployment: {AZURE_OPENAI_CHAT_DEPLOYMENT_NAME}")

✅ Loaded settings from '../0_setup/.foundry_config.json'.
📌 Foundry: hyo-msfoundryv1-pjt1-resource | Region: swedencentral
📌 Project endpoint: https://hyo-ai-foundry-pjt1-resource.services.ai.azure.com/api/projects/hyo-ai-foundry-pjt1
📌 Chat deployment: gpt-4.1


## Part 1: Connect to SK Orchestrator Backend

The evaluation pipeline consumes the **SK Orchestrator Backend** (`sk_backend/`) — a FastAPI service that wraps Semantic Kernel, XML context loading, and OpenTelemetry tracing behind three endpoints:

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/health` | `GET` | Service health check |
| `/classify` | `POST` | Intent classification only |
| `/chat` | `POST` | Full pipeline: intent → XML context → LLM answer |

```
┌─────────────────────────────────────────────────┐
│  This Notebook (Evaluation Pipeline)             │
│                                                  │
│  httpx ──► POST /classify  (intent only)         │
│        ──► POST /chat      (full pipeline)       │
│        ──► GET  /health    (connection check)    │
│                                                  │
│            ▼                                     │
│  ┌──────────────────────────────────────┐        │
│  │  SK Orchestrator Backend (FastAPI)    │        │
│  │  Semantic Kernel + OpenTelemetry      │        │
│  │  → Application Insights              │        │
│  └──────────────────────────────────────┘        │
└─────────────────────────────────────────────────┘
```

> **Prerequisite**: Kill any existing instance and (re)start the backend:
> ```bash
> kill $(lsof -ti :8000) 2>/dev/null; sleep 1
> cd 4_observability/sk_backend && nohup uvicorn sk_orchestrator.main:app --host 0.0.0.0 --port 8000 > sk_backend.log 2>&1 &
> ```


In [14]:
# Connect to SK Orchestrator Backend
import httpx

SK_BACKEND_URL = os.environ.get("SK_BACKEND_URL", "http://localhost:8000")

try:
    health = httpx.get(
        f"{SK_BACKEND_URL}/health", timeout=10.0
    ).json()
    print(f"✅ SK Backend connected: {SK_BACKEND_URL}")
    print(f"   Status:  {health['status']}")
    print(f"   Service: {health['service']}")
    print(f"   Model:   {health['model']}")
except httpx.ConnectError:
    raise ConnectionError(
        f"❌ Cannot connect to SK Backend at {SK_BACKEND_URL}\n"
        "   Start the backend first:\n"
        "   cd 4_observability/sk_backend && "
        "uvicorn sk_orchestrator.main:app --port 8000"
    )


✅ SK Backend connected: http://localhost:8000
   Status:  healthy
   Service: sk-orchestrator-backend
   Model:   gpt-4.1


## Part 2: Orchestration via SK Backend API

Instead of importing Semantic Kernel directly, this notebook calls the **SK Orchestrator Backend** API endpoints to classify intents and generate LLM answers grounded in XML context files.

### API Endpoints Used

| Endpoint | Purpose | Response Keys |
|----------|---------|---------------|
| `POST /classify` | Intent classification | `intent`, `confidence`, `reasoning` |
| `POST /chat` | Full pipeline (intent → XML context → LLM answer) | `query`, `intent`, `confidence`, `agent`, `method`, `context_source`, `answer` |

### Pipeline Flow (via API)

| Step | Description | API Call |
|------|-------------|---------|
| 1 | Intent Classification | `POST /classify` |
| 2 | XML Context Selection + LLM Answer | `POST /chat` |
| 3 | Evaluation Dataset Construction | Combine API responses with CSV ground truth |

### Agent → XML Context Mapping (handled by backend)

| Agent | Method | Intent | XML Context |
|-------|--------|--------|-------------|
| `productAgent` | `search_products` | `product_search` | `product_contexts.xml` |
| `recommendAgent` | `search_recsys` | `recommendation` | `recommendation_contexts.xml` |
| `policyAgent` | `search_policy` | `policy` | (default response) |


In [8]:
# Intent → Agent mapping (ground truth from logs, used by evaluators)
INTENT_AGENT_MAP: Dict[str, Dict[str, str]] = {
    "product_search": {
        "agent": "productAgent",
        "method": "search_products",
    },
    "recommendation": {
        "agent": "recommendAgent",
        "method": "search_recsys",
    },
    "policy": {
        "agent": "policyAgent",
        "method": "search_policy",
    },
    "beauty": {
        "agent": "beautyAgent",
        "method": "search_beauty",
    },
}

print("✅ Intent → Agent mapping defined")
print(f"   Supported intents: {list(INTENT_AGENT_MAP.keys())}")


✅ Intent → Agent mapping defined
   Supported intents: ['product_search', 'recommendation', 'policy', 'beauty']


In [9]:
# API wrapper functions for SK Backend


async def api_classify(query: str) -> dict:
    """
    Call POST /classify to classify user intent.

    Parameters:
    query (str): User query text.

    Returns:
    dict: {"intent", "confidence", "reasoning"}
    """
    async with httpx.AsyncClient(timeout=120.0) as client:
        resp = await client.post(
            f"{SK_BACKEND_URL}/classify",
            json={"query": query},
        )
        resp.raise_for_status()
        return resp.json()


async def api_chat(query: str) -> dict:
    """
    Call POST /chat to run the full pipeline.

    Parameters:
    query (str): User query text.

    Returns:
    dict: {"query", "intent", "confidence", "agent",
           "method", "context_source", "answer"}
    """
    async with httpx.AsyncClient(timeout=120.0) as client:
        resp = await client.post(
            f"{SK_BACKEND_URL}/chat",
            json={"query": query},
        )
        resp.raise_for_status()
        return resp.json()


print("✅ API wrapper functions defined")
print(f"   api_classify(query) → POST {SK_BACKEND_URL}/classify")
print(f"   api_chat(query)     → POST {SK_BACKEND_URL}/chat")


✅ API wrapper functions defined
   api_classify(query) → POST http://localhost:8000/classify
   api_chat(query)     → POST http://localhost:8000/chat


In [17]:
# Test POST /classify endpoint
test_classify = await api_classify(
    "이니스프리 수퍼 화산송이 모공 마스크는?"
)

print("🧪 /classify Endpoint Test")
print(f"   Intent:     {test_classify['intent']}")
print(f"   Confidence: {test_classify['confidence']:.2f}")
print(f"   Reasoning:  {test_classify['reasoning']}")


🧪 /classify Endpoint Test
   Intent:     product_search
   Confidence: 0.95
   Reasoning:  The user is asking about a specific product, '이니스프리 수퍼 화산송이 모공 마스크', which indicates a product search intent.


In [18]:
# Test POST /chat endpoint
test_chat = await api_chat(
    "여드름 피부에 좋은 제품 추천해줘"
)

print("🧪 /chat Endpoint Test")
print(f"   Intent:  {test_chat['intent']}")
print(f"   Agent:   {test_chat['agent']}.{test_chat['method']}")
print(f"   Context: {test_chat['context_source']}")
print(f"   Answer:  {test_chat['answer'][:200]}...")


🧪 /chat Endpoint Test
   Intent:  recommendation
   Agent:   recommendAgent.search_recsys
   Context: /afh/code/agent-operator-lab/4_observability/contexts/recommendation_contexts.xml
   Answer:  여드름 피부(트러블성 피부)에 가장 적합한 제품은 아래와 같습니다:

1. **코스알엑스 AHA/BHA 클래리파잉 트리트먼트 토너**
   - **추천 이유:**  
     AHA와 BHA 성분이 함유되어 있어 각질 제거와 모공 속 피지, 블랙헤드, 트러블 케어에 효과적입니다. 저농도 설계로 매일 사용해도 자극이 적으며, 판테놀과 나이아신아마이드가 피부 ...


In [12]:
# Orchestration wrappers — match the original pipeline function signatures


async def classify_intent(query: str) -> dict:
    """
    Classify intent via SK Backend API.

    Parameters:
    query (str): User query text.

    Returns:
    dict: Intent classification result from POST /classify.
    """
    return await api_classify(query)


async def run_pipeline(query: str) -> dict:
    """
    Run the full pipeline via SK Backend API.

    Parameters:
    query (str): User query text.

    Returns:
    dict: Full pipeline result from POST /chat.
    """
    return await api_chat(query)


print("✅ Orchestration wrappers defined")
print("   classify_intent(query) → POST /classify")
print("   run_pipeline(query)    → POST /chat")


✅ Orchestration wrappers defined
   classify_intent(query) → POST /classify
   run_pipeline(query)    → POST /chat


In [13]:
# Load golden query set
GOLDEN_QUERIES_PATH = "./log/golden_user_query_list.jsonl"

golden_queries: List[Dict[str, str]] = []
with open(GOLDEN_QUERIES_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        golden_queries.append(json.loads(line.strip()))

print(f"✅ Loaded {len(golden_queries)} golden queries")
print(f"   from {GOLDEN_QUERIES_PATH}")

from collections import Counter
intent_dist = Counter(q["expected_intent"] for q in golden_queries)
for intent, count in intent_dist.most_common():
    print(f"   {intent}: {count}")

print(f"\n--- Sample Queries ---")
for q in golden_queries[:3]:
    print(f"   [{q['expected_intent']:18s}] {q['query'][:60]}")


✅ Loaded 50 golden queries
   from ./log/golden_user_query_list.jsonl
   product_search: 20
   recommendation: 15
   policy: 10
   beauty: 5

--- Sample Queries ---
   [product_search    ] 마몽드 플래시토닝 데이지 리퀴드 마스크는 어떤 피부 타입에 좋아?
   [product_search    ] 이니스프리 수퍼 화산송이 모공 마스크 성분이 궁금해요
   [product_search    ] 설화수 윤조에센스 사용방법 알려줘


## Part 3: Build Evaluation Dataset from Golden Queries

Sends the golden query set (`golden_user_query_list.jsonl`) to the SK Backend via `POST /chat`, collects the LLM responses along with the XML context used, and saves the complete evaluation dataset as `llm_result_list.jsonl`.

### Pipeline

```
golden_user_query_list.jsonl     POST /chat (SK Backend)     llm_result_list.jsonl
─────────────────────────── ──► ───────────────────────── ──► ──────────────────────
50 queries with labels           Intent → XML → LLM          Predictions + context
(expected_intent/agent/method)                                + response for eval
```

### Output Fields in `llm_result_list.jsonl`

| Field | Source | Used By |
|-------|--------|---------|
| `query` | Golden query set | All evaluators |
| `expected_intent/agent/method` | Golden labels | Step 1-3 custom evaluators |
| `predicted_intent/agent/method` | SK Backend `/chat` response | Step 1-3 custom evaluators |
| `context` | XML file matching predicted intent | Retrieval + Groundedness |
| `response` | LLM-generated answer | Groundedness + Similarity |
| `ground_truth` | (empty — no human-written answer) | Similarity (skipped if empty) |


In [14]:
# Load XML context files for evaluation (same files the backend uses)
import xml.etree.ElementTree as ET

PRODUCT_CONTEXTS_PATH = "./contexts/product_contexts.xml"
RECOMMENDATION_CONTEXTS_PATH = "./contexts/recommendation_contexts.xml"

_xml_contexts: Dict[str, str] = {}
for label, path in [
    ("product_search", PRODUCT_CONTEXTS_PATH),
    ("recommendation", RECOMMENDATION_CONTEXTS_PATH),
]:
    tree = ET.parse(path)
    _xml_contexts[label] = ET.tostring(
        tree.getroot(), encoding="unicode"
    )

print("✅ XML context files loaded for evaluation")
for k, v in _xml_contexts.items():
    print(f"   {k}: {len(v):,} chars")


✅ XML context files loaded for evaluation
   product_search: 20,275 chars
   recommendation: 28,297 chars


In [15]:
# Run golden queries through SK Backend and save results
LLM_RESULT_PATH = "./log/llm_result_list.jsonl"

print(f"🔄 Sending {len(golden_queries)} golden queries to SK Backend...")

llm_results: List[Dict[str, Any]] = []
for i, gq in enumerate(golden_queries):
    try:
        chat_result = await api_chat(gq["query"])
        predicted_intent = chat_result["intent"]
        predicted_agent = chat_result["agent"]
        predicted_method = chat_result["method"]
        llm_response = chat_result["answer"]

        # XML context matching the predicted intent
        xml_context = _xml_contexts.get(
            predicted_intent, ""
        )
    except Exception as e:
        print(f"   ⚠️ [{i+1}] Error: {e}")
        predicted_intent = "unknown"
        predicted_agent = "unknown"
        predicted_method = "unknown"
        llm_response = ""
        xml_context = ""

    llm_results.append({
        "query": gq["query"],
        # Golden labels (ground truth)
        "expected_intent": gq["expected_intent"],
        "expected_agent": gq["expected_agent"],
        "expected_method": gq["expected_method"],
        # SK Backend predictions
        "predicted_intent": predicted_intent,
        "predicted_agent": predicted_agent,
        "predicted_method": predicted_method,
        # XML context for Retrieval/Groundedness
        "context": xml_context,
        # LLM answer
        "response": llm_response,
        # No human-written ground truth
        "ground_truth": "",
    })

    if (i + 1) % 10 == 0:
        print(f"   Processed {i + 1}/{len(golden_queries)}")

# Save results
with open(LLM_RESULT_PATH, 'w', encoding='utf-8') as f:
    for item in llm_results:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

filled = sum(1 for r in llm_results if r["context"])
print(f"\n✅ {len(llm_results)} results saved to {LLM_RESULT_PATH}")
print(f"   Records with XML context: {filled}/{len(llm_results)}")


🔄 Sending 50 golden queries to SK Backend...


   Processed 10/50
   Processed 20/50
   Processed 30/50
   Processed 40/50
   Processed 50/50

✅ 50 results saved to ./log/llm_result_list.jsonl
   Records with XML context: 30/50


In [16]:
# Verify the evaluation dataset
EVAL_DATA_PATH = "./log/llm_result_list.jsonl"

print(f"📊 Evaluation dataset: {EVAL_DATA_PATH}")
print(f"   Total records: {len(llm_results)}")

from collections import Counter

print("\nPredicted intent distribution:")
pred_dist = Counter(r["predicted_intent"] for r in llm_results)
for intent, count in pred_dist.most_common():
    print(f"   {intent}: {count}")

print("\nIntent match preview (first 10):")
for r in llm_results[:10]:
    match = "✅" if (
        r["predicted_intent"] == r["expected_intent"]
    ) else "❌"
    print(
        f"   {match} expected={r['expected_intent']:18s}"
        f" predicted={r['predicted_intent']:18s}"
        f" | {r['query'][:50]}"
    )


📊 Evaluation dataset: ./log/llm_result_list.jsonl
   Total records: 50

Predicted intent distribution:
   product_search: 15
   recommendation: 15
   beauty: 10
   policy: 10

Intent match preview (first 10):
   ✅ expected=product_search     predicted=product_search     | 마몽드 플래시토닝 데이지 리퀴드 마스크는 어떤 피부 타입에 좋아?
   ✅ expected=product_search     predicted=product_search     | 이니스프리 수퍼 화산송이 모공 마스크 성분이 궁금해요
   ❌ expected=product_search     predicted=beauty             | 설화수 윤조에센스 사용방법 알려줘
   ✅ expected=product_search     predicted=product_search     | 라네즈 워터 슬리핑 마스크 가격이 얼마야?
   ❌ expected=product_search     predicted=beauty             | 헤라 블랙 쿠션 21N1 바닐라 호수가 나한테 맞을까?
   ✅ expected=product_search     predicted=product_search     | 이니스프리 그린티 씨드 세럼 리뷰 평점 알려줘
   ✅ expected=product_search     predicted=product_search     | 마몽드 리퀴드 마스크 민감성 피부에도 사용 가능한가요?
   ✅ expected=product_search     predicted=product_search     | 설화수 진설크림이랑 진설아이크림 차이가 뭐야?
   ✅ expected=product_search     predicted=product_sear

## Part 4: Evaluation Step 1 – Intent Classification (Exact Match)

The first evaluation stage measures the accuracy of **Intent Classification**.

It compares the intent classified by SK against the intent recorded in the actual logs (derived from AGENT_NAME) using **exact matching**.

| Metric | Description |
|--------|-------------|
| `intent_accuracy` | 1.0 if predicted == expected, else 0.0 |
| `intent_match` | "PASS" or "FAIL" |


In [ ]:
# Custom Evaluator: Intent Classification Accuracy (Exact Match)
import sys
from IPython.display import display
from azure.ai.evaluation import evaluate


# Save original stdout — evaluate() may redirect it
_original_stdout = sys.stdout


class IntentAccuracyEvaluator:
    """
    Custom evaluator for intent classification accuracy.

    Compares predicted intent against expected intent using exact matching.
    Returns accuracy score (1.0 or 0.0) and pass/fail label.
    """

    def __init__(self):
        self._name = "intent_accuracy"

    def __call__(
        self,
        *,
        predicted_intent: str,
        expected_intent: str,
        **kwargs,
    ) -> Dict[str, Any]:
        is_match = predicted_intent.strip().lower() == (
            expected_intent.strip().lower()
        )
        return {
            "intent_accuracy": 1.0 if is_match else 0.0,
            "intent_match": "PASS" if is_match else "FAIL",
            "predicted": predicted_intent,
            "expected": expected_intent,
        }


# Run intent classification evaluation
intent_evaluator = IntentAccuracyEvaluator()

intent_result = evaluate(
    data=EVAL_DATA_PATH,
    evaluators={"intent_accuracy": intent_evaluator},
    evaluator_config={
        "intent_accuracy": {
            "column_mapping": {
                "predicted_intent": "${data.predicted_intent}",
                "expected_intent": "${data.expected_intent}",
            }
        }
    },
    azure_ai_project=AZURE_AI_PROJECT_ENDPOINT,
    output_path="./log/eval_step1_intent.json",
)

# Restore stdout after evaluate()
sys.stdout = _original_stdout

metrics = intent_result.get("metrics", {})
display({
    "title": "Step 1: Intent Classification Evaluation",
    "metrics": metrics,
})
metrics

EvaluationException: (InternalError) (UserError) Service invocation failed!
Request: POST eastus2.api.azureml.ms/assetstore/v1.0/temporaryDataReference/createOrGet
Status Code: 403 Forbidden
Error Code: UserError/Auth/Authorization/ResourceMsiTokenDoesntHavePermissionsOnStorage
Reason Phrase: Foundry project MSI hyo-ai-foundry-pjt1-resource/hyo-ai-foundry-pjt1 doesn't have appropriate permis
Response Body: {
  "error": {
    "code": "UserError",
    "severity": null,
    "message": "Foundry project MSI hyo-ai-foundry-pjt1-resource/hyo-ai-foundry-pjt1 doesn't have appropriate permissions on the storage account hyochoinewstorageaccount",
    "messageFormat": "Foundry project MSI {foundryAccountName}/{projectName} doesn't have appropriate permissions on the storage account {storageAccount}",
    "messageParameters": {
      "foundryAccountName": "hyo-ai-foundry-pjt1-resource",
      "projectName": "hyo-ai-foundry-pjt1",
      "storageAccount": "hyochoinewstorageaccount"
    },
    "r

## Part 5: Evaluation Step 2 – Agent Relevance

The second evaluation stage checks whether the **correct agent was invoked** based on the classified intent.

For example, a `product_search` intent should route to `productAgent`, and a `recommendation` intent should route to `recommendAgent`.


In [ ]:
# Custom Evaluator: Agent Relevance
class AgentRelevanceEvaluator:
    """
    Custom evaluator for agent routing accuracy.

    Checks whether the predicted agent matches the expected agent
    based on the intent-to-agent mapping.
    """

    def __init__(self, intent_agent_map: Dict[str, Dict[str, str]]):
        self._intent_agent_map = intent_agent_map

    def __call__(
        self,
        *,
        predicted_agent: str,
        expected_agent: str,
        predicted_intent: str,
        **kwargs,
    ) -> Dict[str, Any]:
        is_match = predicted_agent.strip().lower() == (
            expected_agent.strip().lower()
        )

        expected_from_map = self._intent_agent_map.get(
            predicted_intent, {}
        ).get("agent", "")
        mapping_consistent = expected_from_map.lower() == (
            predicted_agent.lower()
        )

        return {
            "agent_relevance": 1.0 if is_match else 0.0,
            "agent_match": "PASS" if is_match else "FAIL",
            "mapping_consistent": mapping_consistent,
            "predicted_agent": predicted_agent,
            "expected_agent": expected_agent,
        }


# Run agent relevance evaluation
agent_evaluator = AgentRelevanceEvaluator(INTENT_AGENT_MAP)

agent_result = evaluate(
    data=EVAL_DATA_PATH,
    evaluators={"agent_relevance": agent_evaluator},
    evaluator_config={
        "agent_relevance": {
            "column_mapping": {
                "predicted_agent": "${data.predicted_agent}",
                "expected_agent": "${data.expected_agent}",
                "predicted_intent": "${data.predicted_intent}",
            }
        }
    },
    output_path="./log/eval_step2_agent.json",
)

# Restore stdout after evaluate()
sys.stdout = _original_stdout

metrics = agent_result.get("metrics", {})
display({
    "title": "Step 2: Agent Relevance Evaluation",
    "metrics": metrics,
})
metrics

{'title': 'Step 2: Agent Relevance Evaluation',
 'metrics': {'agent_relevance.agent_relevance': 0.9,
  'agent_relevance.mapping_consistent': 1.0}}

{'agent_relevance.agent_relevance': 0.9,
 'agent_relevance.mapping_consistent': 1.0}

## Part 6: Evaluation Step 3 – Method Relevance

The third evaluation stage verifies whether the **correct method was called** via function calling.

Even if the right agent is selected, this step further validates that the agent's specific method (`search_products`, `search_recsys`, etc.) was appropriate for the query.


In [26]:
# Custom Evaluator: Method Relevance
class MethodRelevanceEvaluator:
    """
    Custom evaluator for function calling method relevance.

    Verifies that the correct method was called within the selected agent.
    Checks both exact match and intent-to-method mapping consistency.
    """

    def __init__(self, intent_agent_map: Dict[str, Dict[str, str]]):
        self._intent_agent_map = intent_agent_map

    def __call__(
        self,
        *,
        predicted_method: str,
        expected_method: str,
        predicted_intent: str,
        **kwargs,
    ) -> Dict[str, Any]:
        is_match = predicted_method.strip().lower() == (
            expected_method.strip().lower()
        )

        expected_from_map = self._intent_agent_map.get(
            predicted_intent, {}
        ).get("method", "")
        mapping_consistent = expected_from_map.lower() == (
            predicted_method.lower()
        )

        return {
            "method_relevance": 1.0 if is_match else 0.0,
            "method_match": "PASS" if is_match else "FAIL",
            "mapping_consistent": mapping_consistent,
            "predicted_method": predicted_method,
            "expected_method": expected_method,
        }


# Run method relevance evaluation
method_evaluator = MethodRelevanceEvaluator(INTENT_AGENT_MAP)

method_result = evaluate(
    data=EVAL_DATA_PATH,
    evaluators={"method_relevance": method_evaluator},
    evaluator_config={
        "method_relevance": {
            "column_mapping": {
                "predicted_method": "${data.predicted_method}",
                "expected_method": "${data.expected_method}",
                "predicted_intent": "${data.predicted_intent}",
            }
        }
    },
    output_path="./log/eval_step3_method.json",
)

# Restore stdout after evaluate()
sys.stdout = _original_stdout

metrics = method_result.get("metrics", {})
display({
    "title": "Step 3: Method Relevance Evaluation",
    "metrics": metrics,
})
metrics

{'title': 'Step 3: Method Relevance Evaluation',
 'metrics': {'method_relevance.method_relevance': 0.9,
  'method_relevance.mapping_consistent': 1.0}}

{'method_relevance.method_relevance': 0.9,
 'method_relevance.mapping_consistent': 1.0}

## Part 7: Evaluation Step 4 – Retrieval & Groundedness (Pre-built)

The fourth evaluation stage assesses the quality of the final generated answer using **Pre-built Evaluators**.

| Evaluator | What it measures | Input Fields | Score Range |
|-----------|-----------------|--------------|-------------|
| `RetrievalEvaluator` | Whether the retrieved XML context is relevant to the query | `query`, `context` | 1-5 |
| `GroundednessEvaluator` | Whether the response is faithful to the XML context (no hallucination) | `query`, `response`, `context` | 1-5 |

> **Note**: `SimilarityEvaluator` is not used because the golden query set does not include human-written ground truth answers. Similarity evaluation requires a reference answer to compare against.


In [ ]:
# Pre-built Evaluators: Retrieval & Groundedness
from azure.ai.evaluation import (
    RetrievalEvaluator,
    GroundednessEvaluator,
    AzureOpenAIModelConfiguration,
)

# Model config for AI-assisted evaluators
model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    azure_deployment=AZURE_OPENAI_CHAT_DEPLOYMENT_NAME,
    api_version=AZURE_OPENAI_API_VERSION,
)

# Initialize pre-built evaluators
retrieval_eval = RetrievalEvaluator(model_config=model_config)
groundedness_eval = GroundednessEvaluator(model_config=model_config)

prebuilt_eval_status = {
    "title": "Pre-built Evaluators Initialized",
    "evaluators": [
        "RetrievalEvaluator (query + context)",
        "GroundednessEvaluator (query + response + context)",
    ],
    "note": "SimilarityEvaluator skipped (no ground_truth)",
}

display(prebuilt_eval_status)
prebuilt_eval_status

In [ ]:
# Run Retrieval & Groundedness evaluation
quality_result = evaluate(
    data=EVAL_DATA_PATH,
    evaluators={
        "retrieval": retrieval_eval,
        "groundedness": groundedness_eval,
    },
    evaluator_config={
        "retrieval": {
            "column_mapping": {
                "query": "${data.query}",
                "context": "${data.context}",
            }
        },
        "groundedness": {
            "column_mapping": {
                "query": "${data.query}",
                "response": "${data.response}",
                "context": "${data.context}",
            }
        },
    },
    output_path="./log/eval_step4_quality.json",
)

# Restore stdout after evaluate()
sys.stdout = _original_stdout

metrics = quality_result.get("metrics", {})
display({
    "title": "Step 4: Retrieval & Groundedness",
    "metrics": metrics,
})
metrics

## Part 8: Evaluation Summary Dashboard

Consolidates all stage-by-stage evaluation results into a unified **Evaluation Summary**.

```
┌─────────────────────────────────────────────────────────────────┐
│                    Evaluation Pipeline                           │
│                                                                  │
│  Step 1        Step 2         Step 3         Step 4              │
│  Intent ──► Agent ──► Method ──► Answer Quality                  │
│  (Exact       (Custom        (Custom        (Retrieval +         │
│   Match)       Eval)          Eval)          Groundedness +      │
│                                              Similarity)         │
│                                                                  │
│  Each step evaluates a specific stage of the SK pipeline         │
│  using traces from Application Insights as ground truth          │
└─────────────────────────────────────────────────────────────────┘
```


In [32]:
# Aggregate all evaluation results into a summary
def extract_metric(result: dict, prefix: str) -> Dict[str, Any]:
    """Extract metrics with a specific prefix from evaluation result."""
    metrics = result.get("metrics", {})
    return {k: v for k, v in metrics.items() if prefix in k.lower()}


summary = {
    "Step 1: Intent Classification": extract_metric(
        intent_result, "intent"
    ),
    "Step 2: Agent Relevance": extract_metric(
        agent_result, "agent"
    ),
    "Step 3: Method Relevance": extract_metric(
        method_result, "method"
    ),
    "Step 4: Retrieval": extract_metric(
        quality_result, "retrieval"
    ),
    "Step 4: Groundedness": extract_metric(
        quality_result, "groundedness"
    ),
}

print("=" * 70)
print("📊 EVALUATION PIPELINE SUMMARY")
print(f"   Golden queries: {len(golden_queries)}")
print(f"   LLM results:   {len(llm_results)}")
print("=" * 70)

for step_name, metrics in summary.items():
    print(f"\n🔹 {step_name}")
    if metrics:
        for k, v in metrics.items():
            if isinstance(v, float):
                print(f"     {k}: {v:.4f}")
            else:
                print(f"     {k}: {v}")
    else:
        print("     (no metrics)")

print("\n" + "=" * 70)
print("📁 Detailed results saved to ./log/eval_step*.json")
print(f"🌐 View in Foundry Portal: {AZURE_AI_PROJECT_ENDPOINT}")


📊 EVALUATION PIPELINE SUMMARY
   Golden queries: 50
   LLM results:   50

🔹 Step 1: Intent Classification
     intent_accuracy.intent_accuracy: 0.9000

🔹 Step 2: Agent Relevance
     agent_relevance.agent_relevance: 0.9000
     agent_relevance.mapping_consistent: 1.0000

🔹 Step 3: Method Relevance
     method_relevance.method_relevance: 0.9000
     method_relevance.mapping_consistent: 1.0000

🔹 Step 4: Retrieval
     retrieval.retrieval: 2.4490
     retrieval.gpt_retrieval: 2.4490
     retrieval.retrieval_prompt_tokens: 8847.5000
     retrieval.retrieval_completion_tokens: 205.1600
     retrieval.retrieval_total_tokens: 9052.6600
     retrieval.binary_aggregate: 0.3800

🔹 Step 4: Groundedness
     groundedness.groundedness: 4.7800
     groundedness.gpt_groundedness: 4.7800
     groundedness.binary_aggregate: 0.9400

📁 Detailed results saved to ./log/eval_step*.json
🌐 View in Foundry Portal: https://hyo-ai-foundry-pjt1-resource.services.ai.azure.com/api/projects/hyo-ai-foundry-pjt1


In [33]:
# Row-level result inspection
print("=" * 70)
print("📋 Row-Level Results (first 5 records)")
print("=" * 70)

rows = intent_result.get("rows", [])
for i, row in enumerate(rows[:5]):
    print(f"\n[{i+1}] Query: {row.get('inputs.query', 'N/A')[:60]}...")
    print(f"    Intent: expected={row.get('inputs.expected_intent', '?')}"
          f" / predicted={row.get('inputs.predicted_intent', '?')}"
          f" → {row.get('outputs.intent_accuracy.intent_match', '?')}")

agent_rows = agent_result.get("rows", [])
for i, row in enumerate(agent_rows[:5]):
    print(f"\n[{i+1}] Agent: expected={row.get('inputs.expected_agent', '?')}"
          f" / predicted={row.get('inputs.predicted_agent', '?')}"
          f" → {row.get('outputs.agent_relevance.agent_match', '?')}")

📋 Row-Level Results (first 5 records)

[1] Query: 마몽드 플래시토닝 데이지 리퀴드 마스크는 어떤 피부 타입에 좋아?...
    Intent: expected=product_search / predicted=product_search → PASS

[2] Query: 이니스프리 수퍼 화산송이 모공 마스크 성분이 궁금해요...
    Intent: expected=product_search / predicted=product_search → PASS

[3] Query: 설화수 윤조에센스 사용방법 알려줘...
    Intent: expected=product_search / predicted=beauty → FAIL

[4] Query: 라네즈 워터 슬리핑 마스크 가격이 얼마야?...
    Intent: expected=product_search / predicted=product_search → PASS

[5] Query: 헤라 블랙 쿠션 21N1 바닐라 호수가 나한테 맞을까?...
    Intent: expected=product_search / predicted=product_search → PASS

[1] Agent: expected=productAgent / predicted=productAgent → PASS

[2] Agent: expected=productAgent / predicted=productAgent → PASS

[3] Agent: expected=productAgent / predicted=beautyAgent → FAIL

[4] Agent: expected=productAgent / predicted=productAgent → PASS

[5] Agent: expected=productAgent / predicted=productAgent → PASS


## Wrap-up

### Evaluation Pipeline Summary

```
golden_user_query_list.jsonl (50 queries with labels)
         │
         ▼
    POST /chat (SK Backend)
         │
         ▼
llm_result_list.jsonl (predictions + XML context + response)
         │
         ├──► Step 1: Intent Accuracy  (custom exact match)
         ├──► Step 2: Agent Relevance  (custom evaluator)
         ├──► Step 3: Method Relevance (custom evaluator)
         ├──► Step 4a: Retrieval       (pre-built, query vs context)
         └──► Step 4b: Groundedness    (pre-built, response vs context)
```

| Step | Evaluator Type | Metric | Description |
|------|---------------|--------|-------------|
| 1 | Custom (Exact Match) | `intent_accuracy` | Intent classification accuracy |
| 2 | Custom | `agent_relevance` | Agent routing accuracy |
| 3 | Custom | `method_relevance` | Function calling method accuracy |
| 4a | Pre-built | `retrieval` | XML context relevance to query (1-5) |
| 4b | Pre-built | `groundedness` | Response grounded in XML context (1-5) |

### Key Takeaways

- **Golden query set** provides stable, labeled ground truth for repeatable evaluation
- **No trace-based dataset construction** — eliminates duplicate/incomplete trace issues
- Custom evaluators compare golden labels against SK Backend predictions
- `RetrievalEvaluator` verifies the correct XML context was selected for each query
- `GroundednessEvaluator` verifies the LLM answer stays faithful to the XML context
- Results are logged to Foundry project for portal review

### Next Steps

1. **Expand golden set**: Add more queries to `golden_user_query_list.jsonl`
2. **Add ground truth answers**: Enable `SimilarityEvaluator` by adding human-written answers
3. **CI/CD integration**: Automate the pipeline as a regression test on model updates
4. **Additional evaluators**: Add `RelevanceEvaluator`, `CoherenceEvaluator` for broader quality metrics


## Additional Resources

- [Azure AI Evaluation SDK Documentation](https://learn.microsoft.com/en-us/azure/ai-foundry/how-to/develop/evaluate-sdk)
- [Custom Evaluators Guide](https://learn.microsoft.com/en-us/azure/ai-foundry/concepts/evaluation-evaluators/custom-evaluators)
- [Semantic Kernel OpenTelemetry](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/observability)
- [Application Insights for AI Apps](https://learn.microsoft.com/en-us/azure/azure-monitor/app/app-insights-overview)
- [Azure AI Foundry Portal](https://ai.azure.com/)